In [71]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from xgboost import XGBClassifier
from scipy.sparse import csr_matrix
from sklearn.metrics import log_loss
import xgboost as xgb
from sklearn.preprocessing import LabelEncoder

In [ ]:
def build_features_advanced(df: pd.DataFrame) -> pd.DataFrame:

    df = df.copy()
    

    df['Upc'] = df['Upc'].fillna('0').astype(str)
    df['FinelineNumber'] = df['FinelineNumber'].fillna('-1').astype(str)
    df['DepartmentDescription'] = df['DepartmentDescription'].fillna('UNKNOWN')
    

    df['is_return'] = (df['ScanCount'] < 0).astype(int)
    df['is_purchase'] = (df['ScanCount'] > 0).astype(int)
    df['scan_abs'] = df['ScanCount'].abs()
    

    weekday_map = {'Monday': 1, 'Tuesday': 2, 'Wednesday': 3, 
                   'Thursday': 4, 'Friday': 5, 'Saturday': 6, 'Sunday': 7}
    df['weekday_num'] = df['Weekday'].map(weekday_map)
    

    df = df.sort_values(['Weekday', 'VisitNumber'])
    df['day_group'] = (df['Weekday'] != df['Weekday'].shift()).cumsum()
    day_first = df.groupby('day_group')['VisitNumber'].transform('first')
    day_last = df.groupby('day_group')['VisitNumber'].transform('last')
    df['time_of_day'] = (df['VisitNumber'] - day_first) / (day_last - day_first + 1)
    

    group = df.groupby('VisitNumber')
    

    features = pd.DataFrame({

        'n_items': group['scan_abs'].sum(),
        'n_transactions': group['ScanCount'].count(),
        'n_unique_upc': group['Upc'].nunique(),
        'n_unique_fineline': group['FinelineNumber'].nunique(),
        'n_unique_departments': group['DepartmentDescription'].nunique(),
        

        'n_returns': group['is_return'].sum(),
        'n_purchases': group['is_purchase'].sum(),

        'weekday': group['weekday_num'].first(),
        'time_of_day': group['time_of_day'].first(),
    })

    features['return_ratio'] = features['n_returns'] / (features['n_transactions'] + 1)
    features['purchase_ratio'] = features['n_purchases'] / (features['n_transactions'] + 1)
    features['avg_items_per_transaction'] = features['n_items'] / (features['n_transactions'] + 1)
    features['density'] = features['n_items'] / (features['n_unique_upc'] + 1)
    features['has_returns'] = (features['n_returns'] > 0).astype(int)
    
    weekday_dummies = pd.get_dummies(features['weekday'], prefix='wd')
    features = pd.concat([features, weekday_dummies], axis=1)
    
    top_departments = df['DepartmentDescription'].value_counts().head(15).index
    for dept in top_departments:
        dept_sum = df[df['DepartmentDescription'] == dept].groupby('VisitNumber')['scan_abs'].sum()
        features[f'dept_{dept}_sum'] = features.index.map(dept_sum).fillna(0)
    
    top_upc = df['Upc'].value_counts().head(30).index
    for upc in top_upc:
        upc_count = df[df['Upc'] == upc].groupby('VisitNumber').size()
        features[f'upc_{upc}'] = features.index.map(upc_count).fillna(0)

    dept_distribution = df.groupby(['VisitNumber', 'DepartmentDescription']).size()
    dept_distribution = dept_distribution / dept_distribution.groupby('VisitNumber').sum()
    
    entropy = dept_distribution.groupby('VisitNumber').apply(
        lambda x: -np.sum(x * np.log2(x + 1e-10))
    ).fillna(0)
    features['entropy_departments'] = features.index.map(entropy).fillna(0)
    
    only_returns = df.groupby('VisitNumber').apply(
        lambda x: (x['is_return'].sum() == x['is_return'].count()) and x['is_return'].sum() > 0
    ).astype(int)
    features['only_returns'] = features.index.map(only_returns).fillna(0)
    
    features = features.fillna(0)
    
    features = features.reset_index()
    
    return features

In [ ]:
def build_features_final(df: pd.DataFrame) -> pd.DataFrame:
    """final"""
    df = df.copy()
    
    df['Upc'] = df['Upc'].fillna('0').astype(str)
    df['FinelineNumber'] = df['FinelineNumber'].fillna('-1').astype(str)
    df['DepartmentDescription'] = df['DepartmentDescription'].fillna('UNKNOWN')
    
    df['is_return'] = (df['ScanCount'] < 0).astype(int)
    df['is_purchase'] = (df['ScanCount'] > 0).astype(int)
    df['scan_abs'] = df['ScanCount'].abs()

    weekday_map = {'Monday': 1, 'Tuesday': 2, 'Wednesday': 3, 
                   'Thursday': 4, 'Friday': 5, 'Saturday': 6, 'Sunday': 7}
    df['weekday_num'] = df['Weekday'].map(weekday_map)
    
    df = df.sort_values(['Weekday', 'VisitNumber'])
    df['day_group'] = (df['Weekday'] != df['Weekday'].shift()).cumsum()
    day_first = df.groupby('day_group')['VisitNumber'].transform('first')
    day_last = df.groupby('day_group')['VisitNumber'].transform('last')
    df['time_of_day'] = (df['VisitNumber'] - day_first) / (day_last - day_first + 1)
    
    group = df.groupby('VisitNumber')
    
    features = pd.DataFrame({
        'VisitNumber': group['ScanCount'].first().index,
        'n_items': group['scan_abs'].sum().values,
        'n_transactions': group['ScanCount'].count().values,
        'n_unique_upc': group['Upc'].nunique().values,
        'n_unique_fineline': group['FinelineNumber'].nunique().values,
        'n_unique_departments': group['DepartmentDescription'].nunique().values,
        'n_returns': group['is_return'].sum().values,
        'n_purchases': group['is_purchase'].sum().values,
        'weekday': group['weekday_num'].first().values,
        'time_of_day': group['time_of_day'].first().values,
        'std_scan': group['ScanCount'].std().fillna(0).values,
        'skew_scan': group['ScanCount'].skew().fillna(0).values,
    })
    
    features['return_ratio'] = features['n_returns'] / (features['n_transactions'] + 1)
    features['purchase_ratio'] = features['n_purchases'] / (features['n_transactions'] + 1)
    features['avg_items'] = features['n_items'] / (features['n_transactions'] + 1)
    features['density'] = features['n_items'] / (features['n_unique_upc'] + 1)
    features['has_returns'] = (features['n_returns'] > 0).astype(int)
    
    weekday_dummies = pd.get_dummies(features['weekday'], prefix='wd')
    features = pd.concat([features, weekday_dummies], axis=1)

    top_departments = df['DepartmentDescription'].value_counts().head(15).index
    for dept in top_departments:
        dept_sum = df[df['DepartmentDescription'] == dept].groupby('VisitNumber')['scan_abs'].sum()
        features[f'dept_{dept}_sum'] = features['VisitNumber'].map(dept_sum).fillna(0)

    top_upc = df['Upc'].value_counts().head(30).index
    for upc in top_upc:
        upc_count = df[df['Upc'] == upc].groupby('VisitNumber').size()
        features[f'upc_{upc}'] = features['VisitNumber'].map(upc_count).fillna(0)
    

    dept_dist = df.groupby(['VisitNumber', 'DepartmentDescription']).size()
    dept_dist = dept_dist / dept_dist.groupby('VisitNumber').sum()
    entropy = dept_dist.groupby('VisitNumber').apply(
        lambda x: -np.sum(x * np.log2(x + 1e-10))
    ).fillna(0)
    features['entropy'] = features['VisitNumber'].map(entropy).fillna(0)
    

    only_returns = df.groupby('VisitNumber').apply(
        lambda x: (x['is_return'].sum() == x['is_return'].count()) and x['is_return'].sum() > 0
    ).astype(int)
    features['only_returns'] = features['VisitNumber'].map(only_returns).fillna(0)
    
    top_dept = df['DepartmentDescription'].value_counts().index[0]
    top_dept_count = df[df['DepartmentDescription'] == top_dept].groupby('VisitNumber').size()
    features['top_dept_ratio'] = features['VisitNumber'].map(top_dept_count).fillna(0) / (features['n_transactions'] + 1)
    
    return features.fillna(0)

In [ ]:

train = pd.read_csv("train.csv")
test = pd.read_csv("test.csv")
sample_submission = pd.read_csv("sample_submission.csv")
train

,TripType,VisitNumber,Weekday,Upc,ScanCount,DepartmentDescription,FinelineNumber
0,999,5,Friday,6.811315e+10,-1,FINANCIAL SERVICES,1000.0
1,30,7,Friday,6.053882e+10,1,SHOES,8931.0
2,30,7,Friday,7.410811e+09,1,PERSONAL CARE,4504.0
3,26,8,Friday,2.238404e+09,2,PAINT AND ACCESSORIES,3565.0
4,26,8,Friday,2.006614e+09,2,PAINT AND ACCESSORIES,1017.0
...,...,...,...,...,...,...,...
647049,39,191346,Sunday,3.239000e+10,1,PHARMACY OTC,1118.0
647050,39,191346,Sunday,7.874205e+09,1,FROZEN FOODS,1752.0
647051,39,191346,Sunday,4.072000e+03,1,PRODUCE,4170.0
647052,8,191347,Sunday,4.190008e+09,1,DAIRY,1512.0


In [ ]:
#Preparing data + feature engineering + encoding target calss
y = train.groupby("VisitNumber")['TripType'].first()
train = train.drop(['TripType'], axis=1)

le = LabelEncoder()
y_encoded = le.fit_transform(y)

all_data = pd.concat([train, test])

features = build_features_final(all_data)
#features = csr_matrix(features.values)
#features
train_features = features[features['VisitNumber'].isin(train['VisitNumber'])]
test_features = features[features['VisitNumber'].isin(test['VisitNumber'])]

X = train_features.drop(columns=['VisitNumber'])
X_test = test_features.drop(columns=['VisitNumber'])


In [ ]:
#Spliting+training
X_train, X_val, y_train, y_val = train_test_split(X, y_encoded, test_size=0.2, random_state=42, stratify=y_encoded)

dtrain = xgb.DMatrix(X_train, label=y_train)
dval = xgb.DMatrix(X_val, label=y_val)

params = {
    'objective': 'multi:softprob',
    'num_class': len(set(y)),
    'max_depth': 6,
    'eta': 0.1,  
    'subsample': 0.8,
    'colsample_bytree': 0.8,
    'eval_metric': 'mlogloss',
    'tree_method': 'hist',
    'nthread': -1,
    'seed': 42
}

model = xgb.train(
    params,
    dtrain,
    num_boost_round=300,
    evals=[(dtrain, 'train'), (dval, 'val')],
    early_stopping_rounds=20,
    verbose_eval=50
)




[0]	train-mlogloss:2.76942	val-mlogloss:2.77497
[50]	train-mlogloss:1.32874	val-mlogloss:1.46860
[100]	train-mlogloss:1.15930	val-mlogloss:1.39666
[150]	train-mlogloss:1.05698	val-mlogloss:1.38089
[200]	train-mlogloss:0.97761	val-mlogloss:1.37833
[215]	train-mlogloss:0.95667	val-mlogloss:1.37885


In [81]:
#prediction + logloss score
pred_xgb = model.predict(xgb.DMatrix(X_val))
#y_pred = model.predict(xgb.DMatrix(X_val))
log_loss(y_val, pred_xgb)

1.3788511487988393